# 01 — Tokens and API Costs

> **Frontend → AI Engineer:** Understanding the units that language models process—and the units we pay for.

## What will we learn?

By the end of this notebook, we will be able to:

1. Explain what a token is.
2. Tokenize text using Python.
3. Compare words, characters, and tokens.
4. Calculate the cost of an LLM request.
5. Inspect token usage from a real API response.
6. Estimate the cost of an AI application at scale.

## The core idea

Large Language Models do not process text as complete sentences or individual words.

Before text reaches the model, a **tokenizer** divides it into smaller pieces called **tokens**.

A token can be:

- A complete word
- Part of a word
- A punctuation mark
- A space combined with text
- A piece of code

Tokens matter for two major reasons:

- **Limits:** Models can process only a limited number of tokens at once.
- **Cost:** API usage is commonly priced using input and output tokens.

In this notebook, we will examine both through small experiments.

## Tokens are not the same as words

Consider this sentence:

> I am learning tokenization!

Before running any code, make a prediction:

- How many characters does it contain?
- How many words does it contain?
- How many tokens do you think it contains?

### My prediction

- **Characters:**  25
- **Words:** 4
- **Tokens:** 7

The number of tokens may not match the number of words because a tokenizer can split words, punctuation, and spaces in different ways.

In [1]:
text = "I am learning tokenization!"

character_count = len(text)
words = text.split()
word_count = len(words)

print("Text:", text)
print("Characters:", character_count)
print("Words:", words)
print("Word count:", word_count)

Text: I am learning tokenization!
Characters: 27
Words: ['I', 'am', 'learning', 'tokenization!']
Word count: 4


In [3]:
import tiktoken

# Load a tokenizer encoding
encoding = tiktoken.get_encoding("o200k_base")

# Convert our text into token IDs
token_ids = encoding.encode(text)

# Decode each token separately so we can inspect its text
token_pieces = [
    encoding.decode([token_id])
    for token_id in token_ids
]

print("Original text:", text)
print("Token IDs:", token_ids)
print("Token pieces:", token_pieces)
print("Token count:", len(token_ids))

Original text: I am learning tokenization!
Token IDs: [40, 939, 7524, 6602, 2860, 0]
Token pieces: ['I', ' am', ' learning', ' token', 'ization', '!']
Token count: 6


In [4]:
examples = [
    "Hello world",
    "Tokenization is fascinating.",
    "unbelievable",
    "console.log('Hello, world!');",
    "नमस्ते दुनिया",
]

for example in examples:
    tokens = encoding.encode(example)
    pieces = [encoding.decode([token_id]) for token_id in tokens]

    print(f"Text: {example!r}")
    print(f"Characters: {len(example)}")
    print(f"Words: {len(example.split())}")
    print(f"Tokens: {len(tokens)}")
    print(f"Token pieces: {pieces}")
    print("-" * 50)

Text: 'Hello world'
Characters: 11
Words: 2
Tokens: 2
Token pieces: ['Hello', ' world']
--------------------------------------------------
Text: 'Tokenization is fascinating.'
Characters: 28
Words: 3
Tokens: 5
Token pieces: ['Token', 'ization', ' is', ' fascinating', '.']
--------------------------------------------------
Text: 'unbelievable'
Characters: 12
Words: 1
Tokens: 3
Token pieces: ['un', 'bel', 'ievable']
--------------------------------------------------
Text: "console.log('Hello, world!');"
Characters: 29
Words: 2
Tokens: 8
Token pieces: ['console', '.log', "('", 'Hello', ',', ' world', '!', "');"]
--------------------------------------------------
Text: 'नमस्ते दुनिया'
Characters: 13
Words: 2
Tokens: 5
Token pieces: ['न', 'म', 'स्त', 'े', ' दुनिया']
--------------------------------------------------


## Input tokens and output tokens

When an application calls an LLM, tokens generally belong to two main categories.

### Input tokens

Input tokens represent information sent to the model, such as:

- System or developer instructions
- The user's prompt
- Previous conversation messages
- Retrieved documents
- Tool results

### Output tokens

Output tokens represent content generated by the model, such as:

- The final answer
- Generated code
- Structured data
- Tool-call arguments

Input and output tokens can have different prices. Output tokens are commonly more expensive because the model must generate them sequentially.

## Basic cost formula

API prices are commonly listed **per one million tokens**.

```text
input cost  = (input tokens  / 1,000,000) × input price
output cost = (output tokens / 1,000,000) × output price

total cost = input cost + output cost

In [5]:
model = "gpt-5.6-luna"

input_tokens = 1_000
output_tokens = 500

input_price_per_million = 0.20
output_price_per_million = 1.20

input_cost = (
    input_tokens / 1_000_000
) * input_price_per_million

output_cost = (
    output_tokens / 1_000_000
) * output_price_per_million

total_cost = input_cost + output_cost

print(f"Model: {model}")
print(f"Input cost:  ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Total cost:  ${total_cost:.6f}")

Model: gpt-5.6-luna
Input cost:  $0.000200
Output cost: $0.000600
Total cost:  $0.000800


In [6]:
request_volumes = [
    1,
    1_000,
    100_000,
    1_000_000,
]

print(f"Cost per request: ${total_cost:.6f}\n")
print(f"{'Requests':>12} | {'Estimated cost':>16}")
print("-" * 32)

for number_of_requests in request_volumes:
    estimated_cost = total_cost * number_of_requests

    print(
        f"{number_of_requests:>12,} | "
        f"${estimated_cost:>15,.6f}"
    )

Cost per request: $0.000800

    Requests |   Estimated cost
--------------------------------
           1 | $       0.000800
       1,000 | $       0.800000
     100,000 | $      80.000000
   1,000,000 | $     800.000000


In [8]:
def calculate_token_cost(
    input_tokens: int,
    output_tokens: int,
    input_price_per_million: float,
    output_price_per_million: float,
) -> dict[str, float]:
    """Calculate the token cost of one LLM request."""

    input_cost = (
        input_tokens / 1_000_000
    ) * input_price_per_million

    output_cost = (
        output_tokens / 1_000_000
    ) * output_price_per_million

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": input_cost + output_cost,
    }


cost_breakdown = calculate_token_cost(
    input_tokens=1_000,
    output_tokens=300,
    input_price_per_million=0.20,
    output_price_per_million=1.20,
)

print(cost_breakdown)

{'input_cost': 0.0002, 'output_cost': 0.00035999999999999997, 'total_cost': 0.00056}


In [9]:
import os
from pathlib import Path

from dotenv import load_dotenv


# VS Code and Jupyter may use different working directories,
# so check both the current directory and its parent.
possible_env_files = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
]

env_file = next(
    (path for path in possible_env_files if path.exists()),
    None,
)

if env_file is None:
    raise FileNotFoundError(
        "Could not find .env. Create it from .env.example."
    )

load_dotenv(env_file)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key or api_key.startswith("sk-..."):
    raise ValueError(
        "OPENAI_API_KEY is missing or still contains the placeholder value."
    )

print(f"API key loaded safely from: {env_file}")

API key loaded safely from: /Users/gunal/llm-101/.env


In [11]:
from openai import OpenAI


client = OpenAI()

prompt = "Explain what an LLM token is in one short sentence."

response = client.responses.create(
    model=model,
    input=prompt,
    reasoning={"effort": "none"},
    max_output_tokens=100,
)

print("Prompt:")
print(prompt)

print("\nModel response:")
print(response.output_text)

Prompt:
Explain what an LLM token is in one short sentence.

Model response:
An LLM token is a small unit of text—such as a word, part of a word, or punctuation—that a language model processes.


In [13]:
usage = response.usage

if usage is None:
    raise RuntimeError(
        "The API response did not include token usage."
    )

print("Input tokens:", usage.input_tokens)
print("Output tokens:", usage.output_tokens)
print("Total tokens:", usage.total_tokens)

print(
    "Cached input tokens:",
    usage.input_tokens_details.cached_tokens,
)

print(
    "Reasoning tokens:",
    usage.output_tokens_details.reasoning_tokens,
)

print(
    "Does input + output equal total?",
    usage.input_tokens + usage.output_tokens
    == usage.total_tokens,
)

Input tokens: 18
Output tokens: 33
Total tokens: 51
Cached input tokens: 0
Reasoning tokens: 0
Does input + output equal total? True


In [14]:
actual_cost = calculate_token_cost(
    input_tokens=usage.input_tokens, # type: ignore
    output_tokens=usage.output_tokens, # type: ignore
    input_price_per_million=input_price_per_million,
    output_price_per_million=output_price_per_million,
)

print(f"Input tokens: {usage.input_tokens}") # pyright: ignore[reportOptionalMemberAccess]
print(f"Output tokens: {usage.output_tokens}") # type: ignore
print(f"Total tokens: {usage.total_tokens}") # type: ignore
print(f"Request cost: ${actual_cost['total_cost']:.8f}")

Input tokens: 18
Output tokens: 33
Total tokens: 51
Request cost: $0.00004320
